# Tutorial: saving the activations of Qwen3-8B

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2MOData/LLM-Activation-Sparsity-Analysis-with-Oracle-Activations-A-Research-and-Experimental-Project/blob/main/notebooks/tutorial_saving_activations.ipynb)

This notebook shows how the project saves Qwen3-8B's activations so that the oracle can read them
later, with the functions of `procom.activations`. It saves a few example prompts, looks inside a
file, checks that the saved activations are the right ones, and feeds them to the oracle of the
layer-window sweep (`aobench_layer_window_sweep.ipynb`).

**How to run it**

1. Open the notebook in Colab: click the badge above, or *File → Open notebook →
   GitHub*, paste the repo URL, pick your branch and select this notebook.
2. Select a GPU: *Runtime → Change runtime type → GPU* (the free T4 is enough).
3. If you are testing a branch, set `BRANCH` in the setup cell below.
4. *Runtime → Run all*. The first run takes ~5 min to download Qwen3-8B (16 GB).

Edits made in Colab are **not** saved to the repo, and a Colab runtime is wiped when it
disconnects: see [`tutorial_colab_oracle.ipynb`](tutorial_colab_oracle.ipynb) for the workflow.

## 0. Setup

This cell clones the repo on Colab, checks out the selected `BRANCH` and installs the dependencies.
Rerun it every time you open a Colab session and after every push on GitHub.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/2MOData/LLM-Activation-Sparsity-Analysis-with-Oracle-Activations-A-Research-and-Experimental-Project.git"
BRANCH = "main"  # put your branch here to test code that is not merged yet

if IN_COLAB:
    # Clone the repo (or update it), then install the procom package and its dependencies
    !git clone -q {REPO_URL} /content/procom 2>/dev/null
    !cd /content/procom && git fetch -q && git checkout -q {BRANCH} && git pull -q
    !pip install -q -e /content/procom
    sys.path.insert(0, "/content/procom/src")  # makes procom importable without restarting

The oracle is used through AObench's code, as in the sweep notebook. AObench is not a pip
package: `install_aobench` clones it at the commit used by the paper's code.

In [ ]:
import torch

from procom.oracle import aobench

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → GPU"
print(torch.cuda.get_device_name(0))

aobench.install_aobench("/content/aobench" if IN_COLAB else "outputs/aobench")

## 1. Loading Qwen3-8B

The oracle is the one of the layer-window sweep, `ceselder/qwen3-8b-ao-v3-best` (from
[Building Better Activation Oracles](https://arxiv.org/abs/2606.02609)). It is a LoRA adapter on
Qwen3-8B itself, attached to the model by AObench in part 6. Saving the activations always uses
the model with every LoRA disabled.

Qwen3-8B in bf16 takes ~16 GB: it does not fit on a T4 (15 GB), so in that case we load it in 8-bit.
**8-bit is fine for this demo, not for the experiments**: quantization changes the activations
(it is even one of the sparsity methods we study), so the oracle's answers can differ from those of
the sweep, which ran in bf16. Files for the experiments are saved from the bf16 model, on a GPU
with more than 20 GB.

In [ ]:
import logging

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = aobench.MODEL_NAME  # Qwen/Qwen3-8B

gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
load_in_8bit = gpu_memory_gb < 20
quantization = BitsAndBytesConfig(load_in_8bit=True) if load_in_8bit else None
# In 8-bit, bitsandbytes logs a harmless "inputs will be cast to float16" warning at every
# forward pass: we hide it (and only bitsandbytes warnings)
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map="auto", quantization_config=quantization
)
device = model.device
model.eval()
torch.set_grad_enabled(False)

n_layers = model.config.num_hidden_layers
print(f"GPU {gpu_memory_gb:.0f} GB | 8-bit: {load_in_8bit} | {n_layers} layers")

## 2. What we save

The activations are read **exactly as AObench reads them for the oracle**, so that a saved file
can replace the activations AObench would read itself:

- **Which activations.** The residual stream at the **output of layer `L`**
  (`model.model.layers[L]`), read with every LoRA **disabled**, i.e. from the original Qwen3-8B.
  "Layer `L`" is the number written in the oracle's prompt (`Layer: 21`).
- **Which layers.** `PROJECT_LAYERS`: layers 16 to 30, the three 5-layer windows chosen after the
  layer-window sweep (`PROJECT_WINDOWS`: 16-20, 21-25, 26-30). Any list of layers works.
- **Which tokens.** The **last 20 tokens** of each prompt by default (`n_last_tokens`; `None`
  keeps every token), as AObench gives the oracle the last 20 tokens. A conversation goes through
  the chat template with thinking disabled, and with the assistant header
  (`<|im_start|>assistant`) added when the last message is the user's, as in AObench.

**Which format.** One `.safetensors` file **per example**, named after its id:

| key | shape | content |
|---|---|---|
| `layer_16` … `layer_30` | `(n_positions, 4096)` | the activations, one row per kept token |
| `input_ids` | `(n_tokens,)` | the whole tokenized prompt |
| `positions` | `(n_positions,)` | indices in `input_ids` of the rows |

Each file also stores metadata (model, layers, quantization, prompt, subject...), and an
`index.jsonl` in the folder lists all the examples. One file per example means that files can
be read, copied or deleted one by one, and an interrupted run restarts where it stopped.

In [ ]:
from pathlib import Path

from procom.activations import (
    PROJECT_LAYERS,
    PROJECT_WINDOWS,
    load_activations,
    read_index,
    save_activations,
)

LAYERS = list(PROJECT_LAYERS)  # any subset of range(36) works, e.g. [21, 22, 23, 24, 25]
N_LAST_TOKENS = 20
OUTPUT_DIR = Path("/content/activations_demo" if IN_COLAB else "../data/activations_demo")

print("windows:", PROJECT_WINDOWS)
print("layers: ", LAYERS)

## 3. Saving a few examples

An example is a dict with an `id` (the file name), a `prompt` (a chat conversation, or raw text)
and any other fields we want to keep, such as the `subject`. `save_activations` runs one forward
pass per batch of prompts and writes one file per example.

`allow_quantized=load_in_8bit`: the function refuses a quantized model unless told otherwise,
so that experiment files cannot come from the 8-bit model by mistake.

In [ ]:
examples = [
    {
        "id": "facts-000",
        "subject": "facts",
        "prompt": [{"role": "user", "content": "The capital of Australia is"}],
    },
    {
        "id": "context-000",
        "subject": "context",
        "prompt": [
            {
                "role": "user",
                "content": "The philosopher who drank hemlock taught a student who founded an "
                "academy. That student's most famous pupil was",
            }
        ],
    },
    {
        "id": "calculation-000",
        "subject": "calculation",
        "prompt": [{"role": "user", "content": "Tom has 17 apples and gives away 9. He now has"}],
    },
    {
        "id": "predictable-000",
        "subject": "predictable",
        "prompt": [{"role": "user", "content": "Monday, Tuesday, Wednesday, Thursday,"}],
    },
]

index_path = save_activations(
    model,
    tokenizer,
    examples,
    OUTPUT_DIR,
    layers=LAYERS,
    n_last_tokens=N_LAST_TOKENS,
    allow_quantized=load_in_8bit,
)
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"{path.name:32} {path.stat().st_size / 1e6:6.2f} MB")

Rerunning the cell skips the examples that already have a file (pass `overwrite=True` to
redo them). The index lists every example of the folder:

In [ ]:
for row in read_index(OUTPUT_DIR):
    print(row)

## 4. What is inside a file

`load_activations` returns an `Activations` object: `acts` maps each layer to its tensor,
`positions` gives the token of each row, and `metadata` describes how the file was made.

In [ ]:
saved = load_activations(OUTPUT_DIR / "context-000.safetensors")

print("layers:   ", saved.layers)
print("shape:    ", tuple(saved.acts[21].shape), saved.acts[21].dtype)
print("positions:", saved.positions.tolist())
print("tokens:   ", saved.tokens(tokenizer))
print()
for key, value in saved.metadata.items():
    print(f"{key:24} {value}")

Only the layers asked for are read from the disk, which is useful when a file holds 15 layers
and an analysis needs one window:

In [ ]:
window = load_activations(OUTPUT_DIR / "context-000.safetensors", layers=PROJECT_WINDOWS[1])
print(window.layers)

## 5. Checking that we saved the right activations

transformers can also return every hidden state (`output_hidden_states=True`). There,
`hidden_states[0]` is the embeddings and `hidden_states[L + 1]` is the output of layer `L`:
the saved `layer_L` must be equal to `hidden_states[L + 1]`, and differ from `hidden_states[L]`
(the input of layer `L`). `adapters_disabled` turns off every LoRA, as during the saving.

In [ ]:
from procom.activations import adapters_disabled, tokenize_prompt

ids = torch.tensor([tokenize_prompt(tokenizer, examples[1]["prompt"])], device=device)
with adapters_disabled(model):
    hidden_states = model(input_ids=ids, output_hidden_states=True).hidden_states

for layer in saved.layers:
    reference = hidden_states[layer + 1][0, saved.positions.to(device)].cpu()
    diff = (saved.acts[layer].float() - reference.float()).abs().max().item()
    assert diff <= 1e-3 * reference.float().abs().max().item(), f"layer {layer}: {diff}"
print("every saved layer matches hidden_states[L + 1]")

input_of_21 = hidden_states[21][0, saved.positions.to(device)].cpu()
print("differs from hidden_states[L]:", not torch.equal(saved.acts[21], input_of_21))

## 6. Feeding the saved activations to the oracle

The oracle reads a window of 5 layers. Its prompt has one `Layer: N` line per layer, each followed
by one ` ?` per position, and the vectors are added, norm-matched, onto the ` ?` tokens at the
output of layer 1 (see the sweep notebook).

Two ways to ask it the same question, both through AObench's own code:

- `aobench.ask_oracle` reads the activations from the model itself, as in the sweep notebook;
- `aobench.ask_oracle_about_activations` takes them from our file instead.

If the file holds the activations AObench reads, both give **the same answer** (decoding is
greedy). The first call attaches the oracle's LoRA to the model, which takes a minute.

In [ ]:
question = "Can you name which person the model is thinking about?"

for window in PROJECT_WINDOWS:
    window = list(window)
    from_model = aobench.ask_oracle(model, tokenizer, examples[1]["prompt"], question, window)
    from_file = aobench.ask_oracle_about_activations(model, tokenizer, saved.acts, question, window)
    print(f"Layers {window[0]}-{window[-1]}  (same answer: {from_model == from_file})")
    print("  read by AObench:", from_model)
    print("  from the file:  ", from_file)

Since the file keeps the token of each row, the oracle can also be asked about some positions
only, here the ` pupil` token alone:

In [ ]:
pupil = saved.tokens(tokenizer).index(" pupil")  # row of the " pupil" token
one_token = {layer: acts[pupil : pupil + 1] for layer, acts in saved.acts.items()}
window = list(PROJECT_WINDOWS[1])  # layers 21-25
print(aobench.ask_oracle_about_activations(model, tokenizer, one_token, question, window))

## 7. Keeping the files

The Colab runtime is wiped when it disconnects, so copy the folder to Google Drive. With 15
layers and 20 tokens, a file is ~2.5 MB: a dataset of 4 subjects × 50 examples is ~0.5 GB.

In [ ]:
SAVE_TO_DRIVE = False  # set to True to copy the folder to your Drive

if IN_COLAB and SAVE_TO_DRIVE:
    import shutil

    from google.colab import drive

    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/procom") / OUTPUT_DIR.name
    shutil.copytree(OUTPUT_DIR, destination, dirs_exist_ok=True)
    print("copied to", destination)